# Topic Modeling

To discover the most common topics talked about by customers, we will implement topic modeling algorithms.
Specifically, we will use and compare BERTopic (https://maartengr.github.io/BERTopic/index.html) and FASTopic (https://github.com/bobxwu/FASTopic). BERTopic creates contextual review embeddings using a Transformer model, reduces their dimensionality, groups semantically related data into clusters, and uses class-based TF-IDF to identify the words that best represent each topic. FASTopic is a neural topic model that uses pretrained Transformer embeddings and optimal transport to learn relationships among documents, topics, and words. It is designed to provide efficient, stable, and interpretable topic discovery. After evaluation, the selected model will generate the recurring topics. Examples would include food quality, customer service, cleanliness, pricing, atmosphere, and wait times. These results will be displayed in the dashboard as ranked topic bars, topic-frequency charts, or lists of representative keywords.

In [ ]:
!pip uninstall torch torchvision torchaudio -y

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [6]:
!pip install bertopic fastopic --quiet

In [3]:
import pandas as pd
import numpy as np
import os
from bertopic import BERTopic
from fastopic import FASTopic
from sentence_transformers import SentenceTransformer
import time

In [4]:
chunks = []
for chunk in pd.read_csv("yelp_reviews_clean.csv", chunksize=5000):
    chunks.append(chunk)
df = pd.concat(chunks, ignore_index=True)
docs = df["text"].astype(str).tolist()
df.head()

,review_id,user_id,business_id,business_name,stars,sentiment,date,text
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,Turning Point of North Wales,3.0,neutral,2018-07-07,"If you decide to eat here, just be aware it is..."
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,Body Cycle Spinning Studio,5.0,positive,2012-01-03,I've taken a lot of spin classes over the year...
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,Kettle Restaurant,3.0,neutral,2014-02-05,Family diner. Had the buffet. Eclectic assortm...
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,Zaika,5.0,positive,2015-01-04,"Wow! Yummy, different, delicious. Our favorite..."
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,Melt,4.0,positive,2017-01-14,Cute interior and owner (?) gave us tour of up...


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

# BERTopic

In [8]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_size = 75000  # docs per outer chunk
os.makedirs("embedding_checkpoints", exist_ok=True)

total_docs = len(docs)
num_chunks = -(-total_docs // chunk_size)
print(f"Total documents: {total_docs}")
print(f"Total chunks to process: {num_chunks}\n")

all_embeddings = []
for chunk_num, i in enumerate(range(0, total_docs, chunk_size), start=1):
    checkpoint_path = f"embedding_checkpoints/chunk_{i}.npy"

    if os.path.exists(checkpoint_path):
        # already done, skip
        print(f"Chunk {chunk_num}/{num_chunks} (docs {i}-{i+chunk_size}): already done, loading from disk")
        chunk_embeds = np.load(checkpoint_path)
    else:
        print(f"Chunk {chunk_num}/{num_chunks} (docs {i}-{i+chunk_size}): encoding...")
        chunk_docs = docs[i:i+chunk_size]
        chunk_embeds = embedding_model.encode(chunk_docs, batch_size=96, show_progress_bar=True)
        np.save(checkpoint_path, chunk_embeds)

    all_embeddings.append(chunk_embeds)

embeddings = np.concatenate(all_embeddings, axis=0)

start = time.time()
bertopic_model = BERTopic(embedding_model=embedding_model, verbose=True)
bertopic_topics, bertopic_probs = bertopic_model.fit_transform(docs, embeddings)
bertopic_time = time.time() - start

print(f"BERTopic took {bertopic_time:.1f} seconds")
print(f"Number of topics: {len(set(bertopic_topics)) - 1}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
bertopic_model.get_topic_info().head(10)
bertopic_model.get_topic(0)

# FASTopic

In [11]:
start = time.time()
fastopic_model = FASTopic(num_topics=len(set(bertopic_topics)) - 1)
fastopic_topic_words, fastopic_doc_topics = fastopic_model.fit_transform(docs)
fastopic_time = time.time() - start

print(f"FASTTopic took {fastopic_time:.1f} seconds")

NameError: name 'bertopic_topics' is not defined

In [ ]:
for i, words in enumerate(set(fastopic_topic_words[:10])):
    print(f"Topic {i}: {words}")

In [ ]:
comparison = pd.DataFrame({
    "Metric": ["Runtime (s)", "Num topics", "Framework"],
    "BERTopic": [bertopic_time, len(set(bertopic_topics)) - 1, "HDBSCAN + c-TF-IDF"],
    "FASTopic": [fastopic_time, len(fastopic_topic_words), "Neural topic model"]
})
comparison